# 05_gold_validation

## Purpose

Validate the Day 4 Gold outputs.

This notebook checks whether the Gold tables exist, contain rows, expose expected schemas, and are ready for reporting.

## Expected checks

- Gold table existence
- row counts
- key null checks
- KPI sanity checks
- dashboard output inspection
- sample row review

## Main idea

Gold validation confirms that the final business-facing outputs are technically correct and explainable.

In [0]:
import sys

repo_src_path = "/Workspace/Users/dirella@startsteps.org/vattenfall-week9-capstone-anna/src"

if repo_src_path not in sys.path:
    sys.path.append(repo_src_path)

from pyspark.sql import Row
from pyspark.sql import functions as F

from validation.gold_checks import count_nulls, count_duplicate_grain

In [0]:
catalog = "vattenfall_dev"
schema = "analytics"

gold_tables = [
    f"{catalog}.{schema}.gold_daily_market_summary",
    f"{catalog}.{schema}.gold_weather_impact_summary",
    f"{catalog}.{schema}.gold_grid_incident_summary",
    f"{catalog}.{schema}.gold_regional_operations_dashboard",
]

In [0]:
validation_rows = []

for table_name in gold_tables:
    df = spark.table(table_name)

    validation_rows.append(
        Row(
            table_name=table_name,
            row_count=df.count(),
            column_count=len(df.columns),
        )
    )

gold_validation_df = spark.createDataFrame(validation_rows)

display(gold_validation_df)

In [0]:
empty_tables = gold_validation_df.filter(F.col("row_count") <= 0)

if empty_tables.count() > 0:
    display(empty_tables)
    raise ValueError("Gold validation failed: one or more Gold tables are empty.")

print("Gold row count validation passed.")

In [0]:
duplicate_check_rows = []

grain_checks = {
    f"{catalog}.{schema}.gold_daily_market_summary": ["report_day", "region"],
    f"{catalog}.{schema}.gold_weather_impact_summary": ["report_day", "region"],
    f"{catalog}.{schema}.gold_grid_incident_summary": ["event_day", "region", "severity_band"],
    f"{catalog}.{schema}.gold_regional_operations_dashboard": ["report_day", "region"],
}

for table_name, grain_columns in grain_checks.items():
    df = spark.table(table_name)

    duplicate_check_rows.append(
        Row(
            table_name=table_name,
            grain=", ".join(grain_columns),
            duplicate_grain_count=count_duplicate_grain(df, grain_columns),
        )
    )

duplicate_check_df = spark.createDataFrame(duplicate_check_rows)

display(duplicate_check_df)

In [0]:
null_check_rows = []

null_checks = {
    f"{catalog}.{schema}.gold_daily_market_summary": ["report_day", "region"],
    f"{catalog}.{schema}.gold_weather_impact_summary": ["report_day", "region"],
    f"{catalog}.{schema}.gold_grid_incident_summary": ["event_day", "region", "severity_band"],
    f"{catalog}.{schema}.gold_regional_operations_dashboard": ["report_day", "region"],
}

for table_name, columns in null_checks.items():
    df = spark.table(table_name)

    for column_name in columns:
        null_check_rows.append(
            Row(
                table_name=table_name,
                column_name=column_name,
                null_count=count_nulls(df, column_name),
            )
        )

null_check_df = spark.createDataFrame(null_check_rows)

display(null_check_df)

In [0]:
market_df = spark.table(f"{catalog}.{schema}.gold_daily_market_summary")
weather_df = spark.table(f"{catalog}.{schema}.gold_weather_impact_summary")
grid_df = spark.table(f"{catalog}.{schema}.gold_grid_incident_summary")
dashboard_df = spark.table(f"{catalog}.{schema}.gold_regional_operations_dashboard")

sanity_rows = [
    Row(
        check_name="negative_market_volume",
        invalid_count=market_df.filter(F.col("total_volume_mwh") < 0).count(),
    ),
    Row(
        check_name="negative_wind_speed",
        invalid_count=weather_df.filter(F.col("avg_wind_speed_kmh") < 0).count(),
    ),
    Row(
        check_name="negative_incident_duration",
        invalid_count=grid_df.filter(F.col("total_duration_minutes") < 0).count(),
    ),
    Row(
        check_name="negative_dashboard_incident_count",
        invalid_count=dashboard_df.filter(F.col("total_incident_count") < 0).count(),
    ),
]

sanity_check_df = spark.createDataFrame(sanity_rows)

display(sanity_check_df)

In [0]:
for table_name in gold_tables:
    print(f"\nSample rows for {table_name}")
    display(spark.table(table_name).limit(20))

In [0]:
print("Gold validation completed.")
print("Gold outputs are ready for views, governance inspection, and dashboard delivery.")